### 실습 문제

여러 후보 문장 중에서 질의(query) 문장과 의미가 가장 비슷한 문장을 임베딩 코사인 유사도로 찾는 **간단한 의미 검색**을 구현하세요.

1. 아래 4개의 한글 문장을 후보로 준비하세요.
   - `"강아지가 공원에서 신나게 뛰어놀고 있다."`
   - `"고양이가 따뜻한 소파에서 잠을 잔다."`
   - `"주식 시장이 큰 폭으로 하락했다."`
   - `"투자자들이 증시 폭락으로 큰 손실을 보았다."`
2. `text-embedding-3-small` 모델과 앞에서 정의한 `get_embeddings()` 함수로 후보 문장과 질의 문장을 임베딩하세요.
3. `cosine_similarity` 로 질의와 각 후보의 유사도를 계산하고, 유사도가 가장 높은 문장과 그 점수를 출력하세요.

### 테스트 입력 예시

- 질의: `"반려동물이 즐겁게 놀고 있다."` → 가장 유사: `"강아지가 공원에서 신나게 뛰어놀고 있다."`
- 질의: `"코스피가 폭락했다."` → 가장 유사: `"투자자들이 증시 폭락으로 큰 손실을 보았다."`

## 정답

In [1]:
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = OpenAI()
embedding_model = "text-embedding-3-small"


def get_embeddings(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(input=texts, model=model)
    return np.array([data.embedding for data in response.data])

In [2]:
# 후보 문장과 질의 문장 준비
candidates = [
    "강아지가 공원에서 신나게 뛰어놀고 있다.",
    "고양이가 따뜻한 소파에서 잠을 잔다.",
    "주식 시장이 큰 폭으로 하락했다.",
    "투자자들이 증시 폭락으로 큰 손실을 보았다.",
]
query = "반려동물이 즐겁게 놀고 있다."

# 후보와 질의를 한 번에 임베딩 (마지막 행이 질의)
embeddings = get_embeddings(candidates + [query])
candidate_vectors = embeddings[:-1]
query_vector = embeddings[-1:].reshape(1, -1)

# 질의와 각 후보의 코사인 유사도 계산
similarities = cosine_similarity(query_vector, candidate_vectors)[0]

# 유사도 순으로 정렬하여 출력
ranked = sorted(zip(candidates, similarities), key=lambda x: x[1], reverse=True)
print(f"질의: {query}\n")
for sentence, score in ranked:
    print(f"{score:.4f}  {sentence}")

best_sentence, best_score = ranked[0]
print(f"\n가장 유사한 문장: {best_sentence} (유사도 {best_score:.4f})")

질의: 반려동물이 즐겁게 놀고 있다.

0.4701  강아지가 공원에서 신나게 뛰어놀고 있다.
0.3536  고양이가 따뜻한 소파에서 잠을 잔다.
0.1173  투자자들이 증시 폭락으로 큰 손실을 보았다.
0.1086  주식 시장이 큰 폭으로 하락했다.

가장 유사한 문장: 강아지가 공원에서 신나게 뛰어놀고 있다. (유사도 0.4701)
